# Singleton warbles vs warbles-in-bouts

Compare warble calls that occur alone versus warble calls that are part of a multi-call bout.

**Bout definition (same as `high_freq_bouts_v3.ipynb`):** within `(date_folder, exp, assigned_location)`, two consecutive warbles belong to the same bout if their silent gap (`this.start - prev.stop`) is in `[MIN_ICI_BOUT_S, MAX_ICI_BOUT_S]`. Gaps outside that window start a new bout.

**Singleton** = warble whose bout has size 1.
**In-bout** = warble whose bout has size ≥ `MIN_BOUT_SIZE` (default 5).

Warbles in 2–4-call bouts are dropped from this comparison so the two groups are well-separated.

Comparisons so far:
- **call duration** (works on Mac and cluster — uses table columns only)
- **peak / end frequency** via librosa STFT (cluster only — needs raw WAVs)

## Setup (cross-platform)

In [ ]:
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Single source of truth for bout-detection thresholds (see
# vocalization_analysis/bouts.py). If you want to tune them, do it there.
from vocalization_analysis.bouts import BOUT_THRESHOLDS, detect_bouts

HOST = platform.system()

if HOST == "Darwin":
    DROPBOX     = Path("/Users/gilyginosar/Dropbox (Personal)/Vocalizations_project")
    PARQUET_DIR = DROPBOX / "Data" / "parquet_cache"
    FIGURES_DIR = DROPBOX / "Figures" / "warble_singletons_vs_bouts"
    SAVE_FIGS   = True
elif HOST == "Linux":
    PARQUET_DIR = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/all_calls/parquet_cache")
    FIGURES_DIR = None
    SAVE_FIGS   = False
else:
    raise RuntimeError(f"Unsupported platform: {HOST}")

if SAVE_FIGS:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name, fmt="pdf"):
    if not SAVE_FIGS:
        return
    fig.savefig(FIGURES_DIR / f"{name}.{fmt}", bbox_inches="tight")

DATES_TO_PLOT = ["2025_03", "2025_07", "2025_10", "2026_02"]

print(f"HOST                  = {HOST}")
print(f"PARQUET_DIR           = {PARQUET_DIR}")
print(f"SAVE_FIGS             = {SAVE_FIGS}")
print(f"warble thresholds     = {BOUT_THRESHOLDS['warble']}")

## Load warbles and assign bout ids

Pool all dates, restrict to `event_type == "warble"`, then run bout detection on each call's silent gap to its predecessor within `(date, exp, location)`. Every warble row ends up with `bout_id` and `bout_size`.

In [ ]:
# Step 1: load + filter to warbles.
parts = []
for date_tag in DATES_TO_PLOT:
    df_d = pd.read_parquet(PARQUET_DIR / f"all_calls_{date_tag}.parquet")
    df_d = df_d[df_d["event_type"] == "warble"]
    parts.append(df_d)
warble = pd.concat(parts, ignore_index=True)

# Step 2: bout detection (thresholds come from BOUT_THRESHOLDS["warble"]).
warble = detect_bouts(warble, "warble")

MIN_BOUT_SIZE = BOUT_THRESHOLDS["warble"]["min_bout_size"]  # for downstream prints / labels

print(f"{len(warble):,} warble calls total")
print()
print("Per-date breakdown (calls):")
print(warble.groupby(["date_folder", "bout_kind"]).size().unstack(fill_value=0))
print()
print(f"Per-date bout count (in_bout = size >= {MIN_BOUT_SIZE}):")
bout_sizes = warble.drop_duplicates("bout_id")[["date_folder", "bout_size"]]
print(
    bout_sizes.groupby("date_folder").agg(
        n_bouts=("bout_size", "size"),
        n_singleton_bouts=("bout_size", lambda s: (s == 1).sum()),
        n_small_bouts=("bout_size", lambda s: ((s >= 2) & (s < MIN_BOUT_SIZE)).sum()),
        n_in_bouts=("bout_size", lambda s: (s >= MIN_BOUT_SIZE).sum()),
    )
)

## Duration: singleton vs in-bout, per date

Per-date overlay of duration densities. "In-bout" pools all warble calls from bouts of size ≥ `MIN_BOUT_SIZE`, regardless of their position. We can split by position (1st vs later) in a follow-up cell if interesting.

In [ ]:
KIND_COLORS = {"singleton": "#457B9D", "in_bout": "#E76F51"}

# Drop a handful of obvious mis-segmentation outliers ( >= 1 s warble durations,
# same treatment as high_freq_bouts_v3.ipynb).
MAX_DUR_S = 1.0

dur = warble[["date_folder", "bout_kind", "duration_sec"]].copy()
n_total   = len(dur)
n_dropped = int((dur["duration_sec"] >= MAX_DUR_S).sum())
dur = dur[dur["duration_sec"] < MAX_DUR_S]
print(f"clipped {n_dropped:,} / {n_total:,} warble durations >= {MAX_DUR_S}s "
      f"({100*n_dropped/n_total:.3f}%)")

# Shared log-bins across dates so panels are visually comparable.
log_dur = np.log10(dur["duration_sec"].clip(lower=1e-3))
bins = np.linspace(log_dur.min(), log_dur.max(), 60)

fig, axes = plt.subplots(
    len(DATES_TO_PLOT), 1,
    figsize=(11, 2.4 * len(DATES_TO_PLOT)),
    sharex=True, sharey=False,
)
if len(DATES_TO_PLOT) == 1:
    axes = [axes]

for ax, date_tag in zip(axes, DATES_TO_PLOT):
    sub = dur[dur["date_folder"] == date_tag]
    if sub.empty:
        ax.set_axis_off()
        continue

    for kind_value in ("singleton", "in_bout"):
        s = sub[sub["bout_kind"] == kind_value]["duration_sec"]
        if s.empty:
            continue
        med = s.median() * 1000
        ax.hist(np.log10(s), bins=bins, density=True,
                color=KIND_COLORS[kind_value], edgecolor="white", linewidth=0.4,
                alpha=0.5,
                label=f"{kind_value} (n={len(s):,}, median={med:.0f} ms)")

    ax.set_title(f"{date_tag}   (n = {len(sub):,} calls)", loc="left", fontsize=10)
    ax.set_ylabel("Density")
    ax.legend(fontsize=9, loc="upper right")
    ax.tick_params(labelbottom=True)

# Time-unit tick labels.
tick_seconds = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
tick_labels  = ["10 ms", "20 ms", "50 ms", "100 ms", "200 ms", "500 ms", "1 s"]
for ax in axes:
    ax.set_xticks(np.log10(tick_seconds))
    ax.set_xticklabels(tick_labels, fontsize=9)

axes[-1].set_xlabel("Call duration")
fig.suptitle("Warble duration: singleton vs in-bout (per date)", y=1.0, fontsize=12)
fig.tight_layout()
save_fig(fig, "warble_duration_singleton_vs_inbout_per_date")
plt.show()

### Summary table

Per-date medians and 25/75 percentiles for the two groups, plus a Mann–Whitney U test (Bonferroni-corrected across the 4 dates).

In [ ]:
from scipy.stats import mannwhitneyu

N_TESTS = len(DATES_TO_PLOT)
rows = []
for date_tag in DATES_TO_PLOT:
    sub = dur[dur["date_folder"] == date_tag]
    s_single = sub.loc[sub["bout_kind"] == "singleton", "duration_sec"].values
    s_inbout = sub.loc[sub["bout_kind"] == "in_bout",   "duration_sec"].values
    if len(s_single) >= 3 and len(s_inbout) >= 3:
        _, p_raw = mannwhitneyu(s_single, s_inbout, alternative="two-sided")
        p_bonf = min(p_raw * N_TESTS, 1.0)
    else:
        p_raw = p_bonf = float("nan")
    rows.append({
        "date_folder":   date_tag,
        "n_singleton":   len(s_single),
        "n_in_bout":     len(s_inbout),
        "med_singleton_ms": round(1000 * np.median(s_single), 1) if len(s_single) else np.nan,
        "med_in_bout_ms":   round(1000 * np.median(s_inbout), 1) if len(s_inbout) else np.nan,
        "p25_singleton_ms": round(1000 * np.percentile(s_single, 25), 1) if len(s_single) else np.nan,
        "p75_singleton_ms": round(1000 * np.percentile(s_single, 75), 1) if len(s_single) else np.nan,
        "p25_in_bout_ms":   round(1000 * np.percentile(s_inbout, 25), 1) if len(s_inbout) else np.nan,
        "p75_in_bout_ms":   round(1000 * np.percentile(s_inbout, 75), 1) if len(s_inbout) else np.nan,
        "p_raw":   p_raw,
        "p_bonf":  p_bonf,
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## Acoustic features — peak vs end frequency (cluster only)

For each call we compute two per-call scalars from an STFT magnitude spectrogram:

- `peak_freq_hz` — frequency at the global energy maximum (sum across frames, then argmax). Single number for the whole call.
- `end_freq_hz`  — median of the per-frame peak-frequency over the **last `END_FRAC`** of frames. Captures how the call ends without being thrown by a single noisy frame.

Warbles are FM calls so the **peak vs end** difference is a rough measure of frequency modulation depth.

Runs on the cluster only (needs raw WAVs at `BASE_PROCESSED_AUDIO`). We subsample `N_PER_GROUP` calls per `(date, kind)` to keep runtime reasonable, and require `meanprob_warble >= WARBLE_PROB_THR` so we don't waste compute on iffy classifier hits.

In [ ]:
import functools
import soundfile as sf
import librosa

if HOST == "Linux":
    BASE_PROCESSED_AUDIO = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio")
else:
    BASE_PROCESSED_AUDIO = None  # raw WAVs aren't on the Mac

# Warble band — wider than the newborn band in explore_calls_xplatform so we
# don't clip the high end of warble peaks.
FREQ_BAND_HZ = (1_000, 60_000)
N_FFT        = 512    # ~4 ms at sr=125 kHz
HOP_LENGTH   = 128    # ~1 ms hop
END_FRAC     = 0.25   # fraction of frames at the end of the call to average for end_freq

def call_wav_path(date_folder, exp, channel, file_num):
    if BASE_PROCESSED_AUDIO is None:
        raise RuntimeError("Raw WAVs aren't accessible on this platform.")
    return (BASE_PROCESSED_AUDIO / date_folder / str(int(exp))
            / "Averaged_wavs_w_annotations"
            / f"channel_{int(channel)}_file_{int(file_num):03d}.wav")

@functools.lru_cache(maxsize=256)
def _wav_samplerate(path_str):
    return sf.info(path_str).samplerate

def load_call_slice(date_folder, exp, channel, file_num, start_sec, stop_sec):
    p = call_wav_path(date_folder, exp, channel, file_num)
    sr = _wav_samplerate(str(p))
    start = int(round(start_sec * sr))
    stop  = int(round(stop_sec  * sr))
    y, _ = sf.read(str(p), start=start, stop=stop, dtype="float32", always_2d=False)
    return y, sr

def peak_and_end_freq(y, sr, band=FREQ_BAND_HZ, n_fft=N_FFT, hop=HOP_LENGTH, end_frac=END_FRAC):
    """Return (peak_freq, end_freq) in Hz.

    peak_freq : frequency at the global STFT energy maximum (sum across frames).
    end_freq  : median per-frame peak frequency over the last `end_frac` of frames.
    Returns (nan, nan) if the call is too short or has no in-band energy.
    """
    if len(y) < n_fft:
        return np.nan, np.nan
    y = y - y.mean()
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop, window="hann"))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    in_band = (freqs >= band[0]) & (freqs <= band[1])
    if not in_band.any():
        return np.nan, np.nan
    S = S[in_band]
    f = freqs[in_band]
    if S.sum() == 0:
        return np.nan, np.nan

    # Global peak: collapse time, then argmax.
    peak = float(f[np.argmax(S.sum(axis=1))])

    # End-of-call peak: per-frame argmax over the last end_frac of frames, then median.
    n_frames = S.shape[1]
    n_end = max(1, int(round(end_frac * n_frames)))
    end_block = S[:, -n_end:]
    frame_energy = end_block.sum(axis=0)
    valid = frame_energy > 0
    if not valid.any():
        return peak, np.nan
    per_frame_peak = f[np.argmax(end_block[:, valid], axis=0)]
    end_freq = float(np.median(per_frame_peak))
    return peak, end_freq

In [ ]:
N_PER_GROUP     = 500     # warbles per (date_folder, bout_kind)
WARBLE_PROB_THR = 0.7     # require meanprob_warble >= this

if HOST != "Linux":
    raise RuntimeError("Acoustic features require raw WAVs — run this cell on the cluster.")

pool = warble[
    (warble["bout_kind"].isin(["singleton", "in_bout"]))
    & (warble["meanprob_warble"] >= WARBLE_PROB_THR)
].copy()
print(f"pool after meanprob_warble >= {WARBLE_PROB_THR}: {len(pool):,} calls")

# Balanced subsample: up to N_PER_GROUP per (date, bout_kind).
sub = (
    pool.groupby(["date_folder", "bout_kind"], group_keys=False)
        .apply(lambda g: g.sample(min(len(g), N_PER_GROUP), random_state=0))
        .reset_index(drop=True)
)
print(f"subsample: {len(sub):,} calls")
print(sub.groupby(["date_folder", "bout_kind"]).size().unstack(fill_value=0))

peaks = np.empty(len(sub))
ends  = np.empty(len(sub))
for i, row in enumerate(sub.itertuples(index=False)):
    y, sr = load_call_slice(row.date_folder, row.exp, row.channel,
                            row.file_num,
                            row.start_time_file_sec, row.stop_time_file_sec)
    peaks[i], ends[i] = peak_and_end_freq(y, sr)
    if (i + 1) % 500 == 0:
        print(f"  {i+1:,}/{len(sub):,}")

sub["peak_freq_hz"] = peaks
sub["end_freq_hz"]  = ends
sub["peak_minus_end_hz"] = sub["peak_freq_hz"] - sub["end_freq_hz"]

print()
print("Median per (date, bout_kind):")
print(
    sub.groupby(["date_folder", "bout_kind"]).agg(
        n=("peak_freq_hz", "count"),
        peak_khz=("peak_freq_hz", lambda s: round(s.median() / 1000, 2)),
        end_khz =("end_freq_hz",  lambda s: round(s.median() / 1000, 2)),
        peak_minus_end_khz=("peak_minus_end_hz", lambda s: round(s.median() / 1000, 2)),
    )
)

In [ ]:
# Per-date overlay: peak_freq (top row) and end_freq (bottom row),
# singleton vs in_bout. Shared bins per row so panels are comparable.
FEATURES = [
    ("peak_freq_hz", "Peak frequency"),
    ("end_freq_hz",  "End frequency"),
]

fig, axes = plt.subplots(
    len(FEATURES), len(DATES_TO_PLOT),
    figsize=(3.2 * len(DATES_TO_PLOT), 3.2 * len(FEATURES)),
    sharex="row", sharey="row",
)

for r, (col, label) in enumerate(FEATURES):
    vals_all = sub[col].dropna() / 1000   # kHz
    if vals_all.empty:
        continue
    bins = np.linspace(vals_all.min(), vals_all.max(), 45)

    for c, date_tag in enumerate(DATES_TO_PLOT):
        ax = axes[r, c]
        d = sub[sub["date_folder"] == date_tag]
        for kind_value in ("singleton", "in_bout"):
            s = d[d["bout_kind"] == kind_value][col].dropna() / 1000
            if s.empty:
                continue
            ax.hist(s, bins=bins, density=True,
                    color=KIND_COLORS[kind_value], edgecolor="white", linewidth=0.4,
                    alpha=0.5,
                    label=f"{kind_value} (n={len(s):,}, med={s.median():.1f} kHz)")
        if r == 0:
            ax.set_title(date_tag, fontsize=10)
        if c == 0:
            ax.set_ylabel(f"{label}\nDensity", fontsize=10)
        if r == len(FEATURES) - 1:
            ax.set_xlabel("kHz")
        ax.legend(fontsize=7, loc="upper right")

fig.suptitle(
    f"Warble peak vs end frequency  (n ≤ {N_PER_GROUP}/group, "
    f"meanprob_warble ≥ {WARBLE_PROB_THR})",
    y=1.0, fontsize=12,
)
fig.tight_layout()
save_fig(fig, f"warble_peak_end_freq_singleton_vs_inbout_thr{int(WARBLE_PROB_THR*100)}")
plt.show()